In [ ]:
import pandas as pd, glob

files = glob.glob("../noaa_data_pipeline/data/ndbc/station=44017/**/*.parquet", recursive=True)
df_44017 = pd.concat([pd.read_parquet(f) for f in sorted(files)], ignore_index=True)
df_44017.head(100)


In [ ]:
# Merge all NDBC stations into one timestamp-based dataframe
files = glob.glob("../noaa_data_pipeline/data/ndbc/station=*/**/*.parquet", recursive=True)
df_all = pd.concat([pd.read_parquet(f) for f in sorted(files)], ignore_index=True)

#df_all["timestamp"] = pd.to_datetime(df_all["timestamp"], errors="coerce")
feature_cols = [c for c in df_all.columns if c not in ["timestamp", "station_id"]]

for col in feature_cols:
    df_all[col] = pd.to_numeric(df_all[col], errors="coerce")

df_merged = (
    df_all.groupby("timestamp", as_index=False)[feature_cols]
    .mean()
    .sort_values("timestamp")
    .reset_index(drop=True)
)

df_merged.head(100)

In [ ]:
print(df_merged.info())
df_merged.isna().sum()

In [ ]:
import matplotlib.pyplot as plt

# replace df_merged with your dataframe name (e.g., df_44017, df_all, etc.)
# = pd.to_datetime(df_merged["timestamp"], errors="coerce")

year_counts = df_merged["timestamp"].dt.year.value_counts(dropna=True).sort_index()

ax = year_counts.plot(kind="bar", figsize=(10, 4))
ax.set_title("Distribution of Timestamps by Year")
ax.set_xlabel("Year")
ax.set_ylabel("Number of rows")
plt.tight_layout()
plt.show()

In [ ]:
df_merged.tail(100)

In [ ]:
df_merged = df_merged[df_merged["timestamp"].dt.year >= 2010]
df_merged = df_merged[df_merged["timestamp"].dt.month > 2]

df_merged.head(100)

In [ ]:
cutoff = pd.to_datetime("2018-01-01")
mask = (df_merged["timestamp"] < cutoff) | (
    (df_merged["timestamp"] >= cutoff) & (df_merged["timestamp"].dt.minute == 0)
)
df_merged = df_merged[mask]

df_merged.head(100)

In [ ]:
print(df_merged.info())
df_merged.isna().sum()

In [ ]:
df_merged = df_merged[["timestamp", "wind_speed_ms", "wind_gust_ms", "air_temp_c", "water_temp_c", "dewpoint_c", "wind_direction_deg", "air_pressure_hpa"]]
df_merged.head(100)


In [ ]:
print(df_merged.info())
df_merged.isna().sum()

In [ ]:
# Ensure timestamp is datetime and sorted
df_plot = df_merged.copy()
df_plot["timestamp"] = pd.to_datetime(df_plot["timestamp"], errors="coerce")
df_plot = df_plot.dropna(subset=["timestamp"]).sort_values("timestamp")

# Monthly % missing per feature
monthly_null_pct = (
    df_plot.set_index("timestamp")
           .drop(columns=["station_id"], errors="ignore")  # if present
           .isna()
           .resample("MS")   # month start
           .mean()
           .mul(100)
)

# Plot
ax = monthly_null_pct.plot(figsize=(14, 6), linewidth=1.8)
ax.set_title("Monthly Missingness by Feature")
ax.set_xlabel("Month")
ax.set_ylabel("Missing (%)")
ax.set_ylim(0, 100)
ax.grid(alpha=0.25)
ax.legend(title="Feature", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()
